In [ ]:
import os
import random
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
from torchcubicspline import(natural_cubic_spline_coeffs, NaturalCubicSpline)

from model_function import *

from pysteps import motion

In [2]:
device = 'cuda'

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
cwd = os.getcwd()
data_dir = ''

In [5]:
train_time_scale= slice('2006','2016')
val_time_scale = slice('2016','2016')
test_time_scale = slice('2017','2018')

In [6]:
paths_to_data = [
    str(data_dir) + '/era5_data/geopotential_500/*.nc',
    str(data_dir) + '/era5_data/temperature_850/*.nc',
    str(data_dir) + '/era5_data/2m_temperature/*.nc',
    str(data_dir) + '/era5_data/10m_u_component_of_wind/*.nc',
    str(data_dir) + '/era5_data/10m_v_component_of_wind/*.nc'
]
const_info_path = [str(data_dir) +  '/era5_data/constants/constants_5.625deg.nc']

# This variable is unclear
levels = ["z","t","t2m","u10","v10"]

assert (
    len(paths_to_data) == len(levels)
), "Paths to different type of data must be same as number of types of observations"

In [7]:
Final_train_data = 0
Final_val_data = 0
Final_test_data = 0
max_lev = []
min_lev = []

In [8]:
def get_batched(train_times,data_train_final,lev):
    for idx,year in enumerate(train_times):
        data_per_year = data_train_final.sel(time=slice(str(year),str(year))).load()
        data_values = data_per_year[lev].values
        if idx == 0:
            train_data = torch.from_numpy(data_values).reshape(
                -1, 1, 1 ,data_values.shape[-2], data_values.shape[-1]
            )
            if year % 4 == 0: train_data = torch.cat((train_data[:236],train_data[240:])) #skipping 29 feb in leap year
        else:
            mid_data = torch.from_numpy(data_values).reshape(
                -1 , 1, 1, data_values.shape[-2], data_values.shape[-1]
            )
            if year % 4 == 0: mid_data = torch.cat((mid_data[:236],mid_data[240:]))#skipping 29 feb in leap year
            train_data = torch.cat([train_data,mid_data],dim=1)
    
    return train_data

In [9]:
def get_train_test_data_without_scales_batched(data_path,train_time_scale,val_time_scale,test_time_scale,lev,spectral):
    data = xr.open_mfdataset(data_path, combine='by_coords')
    #data = data.isel(lat=slice(None, None, -1))
    if lev in ["v","u","r","q","tisr"]:
        data = data.sel(level=500)
    data = data.resample(time="6h").nearest(tolerance="1h") # Setting data to be 6-hour cycles
    data_train = data.sel(time=train_time_scale).load()
    data_val = data.sel(time=val_time_scale).load()
    data_test = data.sel(time=test_time_scale).load()
    data_global = data.sel(time=slice('2006','2018')).load()

    max_val = data_global.max()[lev].values.tolist()
    min_val = data_global.min()[lev].values.tolist()


    data_train_final = (data_train - min_val)/ (max_val - min_val)
    data_val_final = (data_val - min_val)/ (max_val - min_val)
    data_test_final = (data_test - min_val)/ (max_val - min_val)

    time_vals = data_test_final.time.values
    train_times = [i for i in range(2006,2016)]
    test_times = [2017,2018]
    val_times = [2016]

    train_data = get_batched(train_times,data_train_final,lev)
    test_data = get_batched(test_times,data_test_final,lev)
    val_data = get_batched(val_times,data_val_final,lev)

    t = [i for i in range(365*4)]
    time_steps = torch.tensor(t).view(-1,1)
    return train_data,val_data,test_data,time_steps,data.lat.values,data.lon.values,max_val,min_val,time_vals


In [10]:
for idx,data in enumerate(paths_to_data):
    Train_data, Val_data, Test_data, time_steps, lat, lon, mean, std, time_stamp = get_train_test_data_without_scales_batched(
        data,
        train_time_scale,
        val_time_scale,
        test_time_scale,
        levels[idx],
        # args.spectral
        0,
    )  
    max_lev.append(mean)
    min_lev.append(std)
    if idx==0: 
        Final_train_data = Train_data
        Final_val_data = Val_data
        Final_test_data = Test_data
    else:
        Final_train_data = torch.cat([Final_train_data,Train_data],dim=2)
        Final_val_data = torch.cat([Final_val_data,Val_data],dim=2)
        Final_test_data = torch.cat([Final_test_data,Test_data],dim=2)

In [11]:
H,W = Train_data.shape[3],Train_data.shape[4]

In [12]:
batch_size = 8

Train_loader = DataLoader(Final_train_data[2:],batch_size=batch_size,shuffle=False,pin_memory=False)
Val_loader = DataLoader(Final_val_data[2:],batch_size=batch_size,shuffle=False,pin_memory=False)
Test_loader = DataLoader(Final_test_data[2:],batch_size=batch_size,shuffle=False,pin_memory=False)
time_loader = DataLoader(time_steps[2:],batch_size=batch_size,shuffle=False,pin_memory=False)
time_idx_steps = torch.tensor([i for i in range(365*4)]).view(-1,1)
time_idx = DataLoader(time_idx_steps[2:],batch_size=batch_size,shuffle=False,pin_memory=False)

In [ ]:
def flow_field(
    frames, 
    method="proesmans", 
    wrap=8, 
    gain=255.0, 
    **kw
):
    x = np.ascontiguousarray(frames, dtype=np.float64) * gain
    x = np.concatenate([x[..., -wrap:], x, x[..., :wrap]], axis=-1)
    uv = motion.get_method(method)(x, **kw)
    return uv[..., wrap:-wrap]


def fit_velocity_pysteps(
    time_idx, 
    data_loader, 
    Final_train_data, 
    num_years,
    paths_to_data, 
    scale, 
    H, 
    W, 
    types,
    method="proesmans", 
    lag=1, 
    alpha=None, 
    **flow_kw,
):
    C = len(paths_to_data) * (scale + 1)
    out = []
    for idx_steps, _ in zip(time_idx, data_loader):
        t0 = idx_steps[0].item()
        idxs = [t0 - i for i in range(lag, -1, -1)]        # lag=1 -> [t0-1, t0]

        # explizit stapeln + permutieren, NICHT view() (siehe Abschnitt 5)
        f = torch.stack([Final_train_data[j].view(num_years, C, H, W) for j in idxs])
        f = f.permute(1, 2, 0, 3, 4).cpu().numpy()          # (years, C, T, H, W)

        v = np.empty((num_years, 2, C, H, W), dtype=np.float32)
        for y in range(num_years):
            for c in range(C):
                uv = flow_field(f[y, c], method=method, **flow_kw)
                v[y, 0, c], v[y, 1, c] = uv[0], uv[1]
        out.append(v)

    Final_v = (-1) * (np.stack(out)) / 6  # (B, years, 2, C, H, W)
    if alpha is not None:
        Final_v = Final_v * np.asarray(alpha).reshape(1, 1, 1, C, 1, 1)
    np.save(f'initial_velocities/optical_flow_vel_farneback/{types}_{method}_vel.npy', Final_v)
    return Final_v

In [ ]:
lk_kw = dict(
    fd_method="shitomasi",
    fd_kwargs=dict(
        max_corners=2000,
        quality_level=0.001,
        min_distance=2,
        block_size=5,
        buffer_mask=0,
    ),
    lk_kwargs=dict(
        winsize=(7, 7),
        nr_levels=1,
    ),
    decl_scale=2,
    nr_std_outlier=3,
    k_outlier=10,           
    size_opening=0,
    interp_method="idwinterp2d",
    interp_kwargs=dict(k=8, power=1.0),
    dense=True,
    verbose=False,
)

vet_kw = dict(
    smooth_gain=1e3,             
    sectors=((16, 8, 4), (16, 8, 4)),
    first_guess=None,                  
    verbose=False,
)

flow_kw_farneback = dict(
    pyr_scale=0.5,
    levels=1,      
    winsize=5,       
    iterations=5,
    poly_n=5,
    poly_sigma=1.1,
    flags=256,   
    size_opening=0,  
    sigma=0.0,  
)

In [ ]:
scale = 0

fit_velocity_pysteps(
    time_idx=time_idx,
    data_loader=Train_loader,
    Final_train_data=Final_train_data,
    num_years=10,
    paths_to_data=paths_to_data,
    scale=scale,
    H=H,
    W=W,
    types='train_10year_2day_mm',
    method='farneback',
    **flow_kw_farneback,
)
fit_velocity_pysteps(
    time_idx=time_idx,
    data_loader=Val_loader,
    Final_train_data=Final_val_data,
    num_years=1,
    paths_to_data=paths_to_data,
    scale=scale,
    H=H,
    W=W,
    types='val_10year_2day_mm',
    method='farneback',
    **flow_kw_farneback,
)
fit_velocity_pysteps(
    time_idx=time_idx,
    data_loader=Test_loader,
    Final_train_data=Final_test_data,
    num_years=2,
    paths_to_data=paths_to_data,
    scale=scale,
    H=H,
    W=W,
    types='test_10year_2day_mm',
    method='farneback',
    **flow_kw_farneback,
)

In [ ]:
vel = np.load("initial_velocities/optical_flow_vel_farneback/val_10year_2day_mm_farneback_vel.npy")   # (183, 10, 2, 5, 32, 64)
print(vel.shape)

t, yr, q = 182, 0, 4        # timestep, year, quantity (0 = z)
vx = vel[t, yr, 0, q]      # (32, 64)
vy = vel[t, yr, 1, q]

lat = np.linspace(87.1875, -87.1875, 32)   # ERA5/WeatherBench: descending
lon = np.linspace(0, 354.375, 64)
LON, LAT = np.meshgrid(lon, lat)

mag = np.hypot(vx, vy)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.pcolormesh(LON, LAT, mag, shading='auto', cmap='viridis')
s = 2                                        # arrow subsampling
ax.quiver(LON[::s, ::s], LAT[::s, ::s],
          vx[::s, ::s], -vy[::s, ::s],       # note the minus
          color='white', scale=None)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
fig.colorbar(im, label="|v| (grid cells / hour)")
plt.show()

In [ ]:
vel = np.load("initial_velocities/original_vel/original_vel_42/val_10year_2day_mm_vel.npy")   # (183, 10, 2, 5, 32, 64)

t, yr, q = 182, 0, 4        # timestep, year, quantity (0 = z)
vx = vel[t, yr, 0, q]      # (32, 64)
vy = vel[t, yr, 1, q]

lat = np.linspace(87.1875, -87.1875, 32)   # ERA5/WeatherBench: descending
lon = np.linspace(0, 354.375, 64)
LON, LAT = np.meshgrid(lon, lat)

mag = np.hypot(vx, vy)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.pcolormesh(LON, LAT, mag, shading='auto', cmap='viridis')
s = 2                                        # arrow subsampling
ax.quiver(LON[::s, ::s], LAT[::s, ::s],
          vx[::s, ::s], -vy[::s, ::s],       # note the minus
          color='white', scale=None)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
fig.colorbar(im, label="|v| (grid cells / hour)")
plt.show()